# 👕 Virtual Try-On Uygulaması

Bu notebook ile bir model resmine istediğiniz kıyafeti giydirebilirsiniz.

**Kullanılan Model:** IDM-VTON (Image-based Virtual Try-On Network)

**Özellikler:**
- Kıyafetler vücuda doğal bir şekilde oturur
- Kumaş dokusu ve kıvrımlar gerçekçi şekilde korunur
- Üst giyim (t-shirt, gömlek, ceket vb.) için optimize edilmiştir

---

## 📦 Adım 1: Gerekli Kütüphaneleri Yükleyin

In [ ]:
# Gerekli kütüphaneleri yükle
!pip install -q diffusers transformers accelerate torch torchvision pillow gradio

import torch
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
from pathlib import Path

print("✅ Kütüphaneler başarıyla yüklendi!")
print(f"GPU kullanılabilir: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 🎨 Adım 2: HuggingFace Virtual Try-On Modelini Kullanma

En kolay yöntem HuggingFace'in Gradio API'sini kullanmaktır. Bu şekilde model indirmeye gerek kalmaz.

In [ ]:
from gradio_client import Client, handle_file

# Gradio client'ı yükle
!pip install -q gradio_client

print("✅ Gradio client yüklendi!")

## 🖼️ Adım 3: Virtual Try-On Fonksiyonu

In [ ]:
def virtual_try_on(person_image_path, garment_image_path, garment_description=""):
    """
    Virtual try-on işlemi yapar.
    
    Parametreler:
    - person_image_path: Model/kişi resminin yolu (str)
    - garment_image_path: Kıyafet resminin yolu (str)
    - garment_description: Kıyafet açıklaması (opsiyonel, str)
    
    Döndürür:
    - Sonuç resmi
    """
    try:
        # IDM-VTON modelini kullan (HuggingFace Space)
        client = Client("yisol/IDM-VTON")
        
        print("🔄 Virtual try-on işlemi başlatılıyor...")
        print(f"   Model resmi: {person_image_path}")
        print(f"   Kıyafet resmi: {garment_image_path}")
        
        result = client.predict(
            dict({"background": handle_file(person_image_path), "layers": [], "composite": None}),  # Model resmi
            handle_file(garment_image_path),  # Kıyafet resmi
            garment_description,  # Kıyafet açıklaması
            True,  # Otomatik maske kullan
            True,  # Otomatik kırpma kullan
            30,    # Denoising steps
            42,    # Seed (tekrarlanabilirlik için)
            api_name="/tryon"
        )
        
        print("✅ İşlem tamamlandı!")
        return result
        
    except Exception as e:
        print(f"❌ Hata oluştu: {str(e)}")
        print("\nAlternatif olarak yerel model deneyelim...")
        return None

def show_results(person_img_path, garment_img_path, result_path):
    """
    Orijinal ve sonuç resimlerini yan yana gösterir.
    """
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Orijinal model resmi
    person_img = Image.open(person_img_path)
    axes[0].imshow(person_img)
    axes[0].set_title('Orijinal Model', fontsize=14, fontweight='bold')
    axes[0].axis('off')
    
    # Kıyafet resmi
    garment_img = Image.open(garment_img_path)
    axes[1].imshow(garment_img)
    axes[1].set_title('Kıyafet', fontsize=14, fontweight='bold')
    axes[1].axis('off')
    
    # Sonuç resmi
    result_img = Image.open(result_path)
    axes[2].imshow(result_img)
    axes[2].set_title('Sonuç (Virtual Try-On)', fontsize=14, fontweight='bold')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()

print("✅ Fonksiyonlar hazır!")

## 📥 Adım 4: Test Resimleri İndirin (Opsiyonel)

Kendi resimlerinizi kullanabilir veya test için örnek resimler indirebilirsiniz.

In [ ]:
# Test klasörünü oluştur
!mkdir -p test_images

# Örnek resimler için URL'ler (kendi resimlerinizi de yükleyebilirsiniz)
print("📌 Test resimleri için:")
print("   1. Sol menüden 'Files' ikonuna tıklayın")
print("   2. 'Upload' butonuna basın")
print("   3. Model resminizi ve kıyafet resminizi yükleyin")
print("")
print("💡 İpuçları:")
print("   - Model resmi: Tek kişi, dik duruş, düz arka plan ideal")
print("   - Kıyafet resmi: Düz zemin, kıyafet net görünür olmalı")
print("   - Üst giyim (t-shirt, gömlek, kazak) en iyi sonucu verir")

## 🚀 Adım 5: Virtual Try-On Uygulaması

Kendi resimlerinizin yollarını aşağıya yazın ve çalıştırın!

In [ ]:
# 👇 BURAYA KENDİ RESİM YOLLARINIZI YAZIN
PERSON_IMAGE = "/content/model.jpg"      # Model/kişi resminiz
GARMENT_IMAGE = "/content/garment.jpg"   # Kıyafet resminiz
GARMENT_DESC = ""                         # Opsiyonel: "mavi tişört", "çizgili gömlek" gibi

# Virtual try-on işlemini başlat
result = virtual_try_on(PERSON_IMAGE, GARMENT_IMAGE, GARMENT_DESC)

if result:
    print(f"\n💾 Sonuç kaydedildi: {result}")
    
    # Sonuçları göster
    show_results(PERSON_IMAGE, GARMENT_IMAGE, result)
else:
    print("❌ İşlem başarısız oldu.")

## 🎯 Alternatif Yöntem: Yerel Model Kullanımı

HuggingFace API çalışmazsa, modeli yerel olarak indirip kullanabilirsiniz. (Daha fazla GPU belleği gerektirir)

In [ ]:
# Gelişmiş kullanıcılar için: Yerel model yükleme
def setup_local_model():
    """
    IDM-VTON modelini yerel olarak yükler.
    NOT: Bu yöntem daha fazla GPU belleği gerektirir (16GB+)
    """
    print("⚠️  Bu yöntem daha fazla GPU belleği gerektirir.")
    print("🔄 Model indiriliyor... (Bu biraz zaman alabilir)")
    
    try:
        # OOTDiffusion veya benzeri model kullanımı
        !git clone https://github.com/levihsu/OOTDiffusion.git
        %cd OOTDiffusion
        !pip install -r requirements.txt
        
        print("✅ Model kurulumu tamamlandı!")
        print("📖 Kullanım için model dokümantasyonuna bakın.")
        
    except Exception as e:
        print(f"❌ Kurulum hatası: {str(e)}")

# Yerel modeli kurmak isterseniz bu fonksiyonu çağırın:
# setup_local_model()

## 📊 İpuçları ve En İyi Uygulamalar

### ✅ En İyi Sonuçlar İçin:

**Model Resmi:**
- Tek kişi olmalı
- Dik duruş, vücut tam görünür
- Düz, tek renkli arka plan ideal
- Yüksek çözünürlük (512x512 veya daha yüksek)
- Kollar açık/görünür

**Kıyafet Resmi:**
- Düz zemin üzerinde çekilmiş
- Kıyafet net ve tam görünür
- Kırışıklar minimum
- İyi aydınlatma
- Şeffaf/beyaz arka plan ideal

**Desteklenen Kıyafet Türleri:**
- ✅ T-shirt
- ✅ Gömlek
- ✅ Kazak
- ✅ Ceket
- ✅ Üst giyim genel
- ⚠️ Pantolon (sınırlı destek)
- ⚠️ Elbise (bazı modeller destekler)

### 🔧 Sorun Giderme:

1. **Kıyafet vücuda doğru oturmuyor:**
   - Model resminde vücut pozisyonunu kontrol edin
   - Farklı bir model resmi deneyin
   - Denoising steps değerini artırın (30 → 50)

2. **API hatası alıyorsanız:**
   - İnternet bağlantınızı kontrol edin
   - Biraz bekleyip tekrar deneyin
   - Yerel model kurulumunu deneyin

3. **Sonuç kalitesi düşük:**
   - Daha yüksek çözünürlüklü resimler kullanın
   - Model ve kıyafet resimleri iyi aydınlatılmış olmalı
   - Farklı seed değerleri deneyin

---

## 🎉 Başarılar!

Artık virtual try-on uygulamanız hazır. İyi eğlenceler! 🚀